<a href="https://colab.research.google.com/github/MIARD/SMC/blob/main/SMC_CLEAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from __future__ import annotations

import sys, traceback
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from datetime import time
from functools import partial
from ipywidgets import widgets

In [ ]:
main_df = pd.read_excel('SSL_T.xlsx')
# df = pd.read_excel('SSL_T_MR.xlsx')
# df = pd.read_excel('SSL_M.xlsx')

# df = pd.read_excel('Train.xlsx')
# df = pd.read_excel('Test.xlsx')
# df = pd.read_excel('Train_Test.xlsx')

pd.set_option('display.max_rows', None)

# =============================
# Data preparation
# =============================

In [ ]:
def prepare_dataframe(main_df: pd.DataFrame) -> pd.DataFrame:
    df = main_df.copy()

    # Ensure Start/Exit datetime
    df['startDateTime'] = pd.to_datetime(df['Start Date'].astype(str) + ' ' + df['Start Time'].astype(str), errors='coerce')
    df['exitDateTime']  = pd.to_datetime(df['Exit Date'].astype(str) + ' ' + df['Exit Time'].astype(str), errors='coerce')

    # Durations & P/L
    df['Trade Duration (hours)'] = (df['exitDateTime'] - df['startDateTime']).dt.total_seconds() / 3600
    df['Profit/Loss'] = df['Profit'] - df['Loss'] - df['Fee']

    # Time parts
    df['Hour'] = df['startDateTime'].dt.hour
    df['Min'] = df['startDateTime'].dt.minute
    df['15Min'] = df['startDateTime'].dt.strftime('%H:%M')
    df['Month'] = df['startDateTime'].dt.month_name()
    df['Week'] = df['startDateTime'].dt.isocalendar().week.astype(int)
    df['Year'] = df['startDateTime'].dt.year
    df['Month_Year'] = df['Month'] + "-" + df['startDateTime'].dt.to_period('M').astype(str)
    df['Start_Weekday'] = df['startDateTime'].dt.day_name()
    df['Date'] = df['startDateTime'].dt.date
    df['Day_of_Month'] = df['startDateTime'].dt.day
    df['Week_of_Month'] = df['startDateTime'].apply(lambda x: (x.day - 1) // 7 + 1)
    df['Week_of_Month_Name'] = df['Week_of_Month'].astype(str) + '-' + df['Month']

    # Win R only for winning trades
    df['Win R'] = np.where(df['Profit/Loss'] > 0, df['R'], np.nan)

    # Normalize types for filters (robustness)
    if 'Start Time' in df.columns and not np.issubdtype(df['Start Time'].dtype, np.datetime64):
        # If it's already a pytime, keep it; else parse gracefully
        df['Start Time'] = pd.to_datetime(df['Start Time'].astype(str), errors='coerce').dt.time

    return df

In [ ]:
def calculate_trade_metrics_updated(df: pd.DataFrame, group_keys: list[str], chart_type: str) -> pd.DataFrame:
    grouped = df.groupby(group_keys, dropna=False)
    daily_grouped = df.groupby(group_keys + ['Date'], dropna=False)

    summary = grouped.agg(
        Total_Trades=('Profit/Loss', 'count'),
        Total_Win=('Profit/Loss', lambda x: (x > 0).sum()),
        Total_Loss=('Profit/Loss', lambda x: (x < 0).sum()),
        Total_Profit=('Profit', 'sum'),
        Total_Loss_Amount=('Loss', 'sum'),
        Total_Fee=('Fee', 'sum'),
        Total_Time=('Trade Duration (hours)', 'sum'),
        Average_Time=('Trade Duration (hours)', 'mean'),
        Realized_Profit_Loss=('Profit/Loss', 'sum'),
        Average_Profit=('Profit/Loss', 'mean'),
        Max_Profit=('Profit/Loss', 'max'),
        Max_Loss=('Profit/Loss', 'min'),
        R=('R', 'mean'),
        Average_Win_R=('Win R', 'mean')
    ).reset_index()

    # Daily metrics for Win/Loss days
    daily_profit = daily_grouped['Profit/Loss'].sum().reset_index(name='Daily_PL')
    win_days = daily_profit[daily_profit['Daily_PL'] > 0].groupby(group_keys, dropna=False).size()
    loss_days = daily_profit[daily_profit['Daily_PL'] < 0].groupby(group_keys, dropna=False).size()
    avg_win_day_profit = daily_profit[daily_profit['Daily_PL'] > 0].groupby(group_keys, dropna=False)['Daily_PL'].mean()
    avg_daily_profit = daily_profit.groupby(group_keys, dropna=False)['Daily_PL'].mean()
    max_daily_profit = daily_profit.groupby(group_keys, dropna=False)['Daily_PL'].max()

    summary = summary.set_index(group_keys)
    summary['Total_Win_Days'] = win_days
    summary['Total_Loss_Days'] = loss_days
    summary['Average_Win_Days_Profit'] = avg_win_day_profit
    summary['Average_Daily_Profit'] = avg_daily_profit
    summary['Max_Daily_Profit'] = max_daily_profit

    summary = summary.fillna(0).reset_index()
    summary['Win_Rate'] = (summary['Total_Win'] / summary['Total_Trades']).replace([np.inf, -np.inf], 0).fillna(0).round(4)
    return summary

In [ ]:
_DEF_ORDER = [
    'Total_Trades','Total_Win','Total_Loss','Win_Rate','Average_Win_R','R',
    'Total_Profit','Total_Loss_Amount','Realized_Profit_Loss',
    'Total_Win_Days','Total_Loss_Days','Average_Win_Days_Profit'
]


def _format_hover(label: str) -> str:
    """2‑decimal numeric formatting, Win_Rate shown as percent with 2‑dp."""
    return (
        f"<b>%{{x}}</b><br>{label}: %{{y:,.2f}}<br>"
        "Total Trades: %{customdata[0]:,.0f}<br>"
        "Total Wins: %{customdata[1]:,.0f}<br>"
        "Total Losses: %{customdata[2]:,.0f}<br>"
        "Win Rate: %{customdata[3]:.2%}<br>"
        "Avg Win R: %{customdata[4]:.2f}<br>"
        "R Ratio: %{customdata[5]:.2f}<br>"
        "Total Profit: %{customdata[6]:,.2f}<br>"
        "Total Loss: %{customdata[7]:,.2f}<br>"
        "Realized Profit: %{customdata[8]:,.2f}<br>"
        "Win Days: %{customdata[9]:,.0f}<br>"
        "Loss Days: %{customdata[10]:,.0f}<br>"
        "Avg Win Day Profit: %{customdata[11]:,.2f}<extra></extra>"
    )

In [ ]:
def plot_interactive_bar(summary_df: pd.DataFrame, x_label: str, chart_type: str = 'bar') -> go.Figure:
    custom_data = summary_df[_DEF_ORDER].values.tolist()

    def build_trace(name: str, y_vals: pd.Series, visible: bool) -> go.BaseTraceType:
        common = dict(
            x=summary_df[x_label].astype(str),
            y=y_vals,
            name=name,
            text=[f"{v:,.2f}" if isinstance(v, (int,float,np.floating)) else str(v) for v in y_vals],
            textposition='outside' if chart_type=='bar' else 'top center',
            customdata=custom_data,
            hovertemplate=_format_hover(name),
            visible=visible
        )
        if chart_type == 'line':
            return go.Scatter(mode='lines+markers+text', **common)
        return go.Bar(**common)

    traces = [
        build_trace('Total Trades', summary_df['Total_Trades'], True),
        build_trace('Win Trades', summary_df['Total_Win'], True),
        build_trace('Loss Trades', summary_df['Total_Loss'], True),
        build_trace('Total Profit', summary_df['Total_Profit'], False),
        build_trace('Total Loss Amount', summary_df['Total_Loss_Amount'], False),
        build_trace('Realized Profit', summary_df['Realized_Profit_Loss'], False),
        build_trace('Win Days', summary_df['Total_Win_Days'], False),
        build_trace('Loss Days', summary_df['Total_Loss_Days'], False),
        build_trace('Avg Win Day Profit', summary_df['Average_Win_Days_Profit'], False),
        build_trace('Win Rate (%)', summary_df['Win_Rate'] * 100.0, False)
    ]

    dropdown_buttons = [
        dict(label="Trade Counts", method="update",
             args=[{"visible": [True, True, True] + [False]*7},
                   {"title": f"📊 {x_label} Trade Count Summary", "yaxis": {"title": "Count"}}]),
        dict(label="Profit/Loss", method="update",
             args=[{"visible": [False]*3 + [True, True, True] + [False]*4},
                   {"title": f"💰 {x_label} Profit/Loss Summary", "yaxis": {"title": "Amount"}}]),
        dict(label="Win/Loss Days", method="update",
             args=[{"visible": [False]*6 + [True, True, True, False]},
                   {"title": f"📅 {x_label} Win/Loss Day Summary", "yaxis": {"title": "Days"}}]),
        dict(label="Win Rate", method="update",
             args=[{"visible": [False]*9 + [True]},
                   {"title": f"🏆 {x_label} Win Rate Summary", "yaxis": {"title": "Percent"}}])
    ]

    fig = go.Figure(data=traces)
    fig.update_layout(
        title=f"📊 {x_label} Trading Summary",
        xaxis_title=x_label,
        yaxis_title="Count / Amount",
        barmode='group' if chart_type == 'bar' else None,
        bargap=0.2 if chart_type == 'bar' else None,
        bargroupgap=0.1 if chart_type == 'bar' else None,
        xaxis_tickangle=-45,
        hovermode='closest',
        height=700,
        updatemenus=[dict(buttons=dropdown_buttons, direction="down", showactive=True,
                          x=1.0, xanchor="right", y=1.15, yanchor="top")],
        xaxis=dict(ticks="outside", showline=True, mirror=True),
        yaxis=dict(ticks="outside", showline=True, mirror=True)
    )
    return fig

In [ ]:
def plot_interactive_heatmap(stats: dict[str, pd.DataFrame], row_col: list[str]) -> go.Figure:
    def fmt_df(df, fmt):
        return [[fmt(val) if pd.notnull(val) else "" for val in row] for row in df.values]

    z_win = stats['Win_Rate'].values
    z_pl  = stats['Realized_Profit_Loss'].values
    z_cnt = stats['Total_Trades'].values

    text_win   = fmt_df(stats['Win_Rate'], lambda x: f"{x:.2%}")
    text_pl    = fmt_df(stats['Realized_Profit_Loss'], lambda x: f"{x:,.2f}")
    text_count = fmt_df(stats['Total_Trades'], lambda x: f"{int(x)}")

    hover_text = []
    for i, row_val in enumerate(stats['Win_Rate'].index):
        row = []
        for col_val in stats['Win_Rate'].columns:
            trades = int(stats.loc[row_val, ('Total_Trades', col_val)])
            wins   = int(stats.loc[row_val, ('Total_Win', col_val)])
            losses = int(stats.loc[row_val, ('Total_Loss', col_val)])
            pl     = stats.loc[row_val, ('Realized_Profit_Loss', col_val)]
            wr     = stats['Win_Rate'].loc[row_val, col_val]
            if trades > 0:
                row.append(
                    f"{row_col[0]}: {row_val}<br>{row_col[1]}: {col_val}<br>"
                    f"Total Trades: {trades}<br>Win Trades: {wins}<br>Loss Trades: {losses}<br>"
                    f"Total P/L: {pl:,.2f}<br>Win Rate: {wr:.2%}"
                )
            else:
                row.append(f"{row_col[0]}: {row_val}<br>{row_col[1]}: {col_val}<br>No Trades")
        hover_text.append(row)

    def heat(z, text, colorscale, title, visible):
        # Safe z-limits (avoid zmin=zmax issues)
        z_flat = z[np.isfinite(z)]
        if z_flat.size == 0:
            zmin, zmax = 0, 1
        else:
            zmin, zmax = float(np.nanmin(z_flat)), float(np.nanmax(z_flat))
            if zmin == zmax:
                zmin, zmax = zmin - 1e-9, zmax + 1e-9
        return go.Heatmap(
            z=z,
            x=list(stats['Win_Rate'].columns),
            y=list(stats['Win_Rate'].index),
            text=text,
            hovertext=hover_text,
            texttemplate="%{text}",
            hoverinfo='text',
            colorscale=colorscale,
            showscale=True,
            visible=visible,
            zmin=zmin, zmax=zmax,
            colorbar=dict(title=title)
        )

    traces = [
        heat(z_win, text_win, 'Blues', 'Win Rate', True),
        heat(z_pl,  text_pl,  'RdBu',  'Realized_Profit_Loss', False),
        heat(z_cnt, text_count,'Viridis','Total Trades', False)
    ]

    dropdown = [
        dict(label="Win Rate", method="update",
             args=[{"visible": [True, False, False]},
                   {"title": f"📊 {' / '.join(row_col)} Win Rate"}]),
        dict(label="Total P/L", method="update",
             args=[{"visible": [False, True, False]},
                   {"title": f"💰 {' / '.join(row_col)} Realized Profit Loss"}]),
        dict(label="Trade Count", method="update",
             args=[{"visible": [False, False, True]},
                   {"title": f"📈 {' / '.join(row_col)} Trade Count"}])
    ]

    layout = go.Layout(
        title=f"📊 {' / '.join(row_col)} Win Rate",
        xaxis=dict(title=row_col[1], tickangle=90, tickfont=dict(size=10), showline=True, ticks='outside', mirror='ticks', automargin=True),
        yaxis=dict(title=row_col[0], autorange="reversed", showline=True, ticks='outside', mirror='ticks', tickfont=dict(size=10), automargin=True),
        hovermode="closest",
        width=1200,
        updatemenus=[dict(buttons=dropdown, direction="down", showactive=True, x=1.0, xanchor="right", y=1.2, yanchor="top", font=dict(size=12))]
    )
    return go.Figure(data=traces, layout=layout)

In [ ]:
def generate_visualizations_updated(summary_df: pd.DataFrame, group_keys: list[str], chart_type: str = 'bar') -> go.Figure:
    if chart_type in ['bar', 'line'] and len(group_keys) == 1:
        return plot_interactive_bar(summary_df, group_keys[0], chart_type)
    elif chart_type == 'heatmap' and len(group_keys) == 2:
        row_key, col_key = group_keys
        heatmap_ready_df = summary_df.pivot(index=row_key, columns=col_key)
        heatmap_ready_df = heatmap_ready_df.fillna(0)
        return plot_interactive_heatmap(heatmap_ready_df, group_keys)
    else:
        raise ValueError("Invalid chart_type or grouping")

In [ ]:
class TradingDashboard:
    def __init__(self, df: pd.DataFrame):
        self.df_raw = df.copy()
        self.df = df.copy()
        self.summary_df = pd.DataFrame()
        self.last_fig = None
        self.last_summary = None
        self.last_filtered_df = None
        self._build_widgets()

    # ---------- Widgets ----------
    def _build_widgets(self):
        df = self.df
        # Chart/grouping
        self.chart_dropdown = widgets.Dropdown(options=['bar', 'line', 'heatmap'], value='bar', description='Chart Type:')
        self.group1_dropdown = widgets.Dropdown(options=['15Min','Hour','Start_Weekday','Month','Week','Year','Criteria','Date','Day_of_Month','Week_of_Month','Month_Year','Week_of_Month_Name'], description='Group 1:')
        self.group2_dropdown = widgets.Dropdown(options=['Month','15Min','Hour','Start_Weekday','Week','Year','Criteria','Date','Day_of_Month','Week_of_Month','Month_Year','Week_of_Month_Name'], description='Group 2:')
        self.group2_dropdown.layout.display = 'none'
        # self.all_cols = df.columns.tolist()
        # self.group1_dropdown = widgets.Dropdown(options=self.all_cols, description='Group 1:')
        # self.group2_dropdown = widgets.Dropdown(options=self.all_cols, description='Group 2:')
        # Dates
        min_date = pd.to_datetime(df['Start Date'], format='%Y-%m-%d', errors='coerce').min().date()
        max_date = pd.to_datetime(df['Start Date'], format='%Y-%m-%d', errors='coerce').max().date()
        self.start_date_picker = widgets.DatePicker(value=min_date, description='From:')
        self.end_date_picker   = widgets.DatePicker(value=max_date, description='To:')
        self.date_slider = widgets.SelectionRangeSlider(options=[d.date() for d in pd.date_range(min_date, max_date)],
                                                        index=(0, len(pd.date_range(min_date, max_date)) - 1),
                                                        description='Date Range:', layout={'width': '100%'})

        # Time
        time_options = [time(h, m) for h in range(24) for m in [0, 15, 30, 45]]
        self.time_options = time_options
        self.start_time_picker = widgets.SelectionSlider(options=time_options, value=time(0, 0), description='From Time:')
        self.end_time_picker   = widgets.SelectionSlider(options=time_options, value=time(23, 45), description='To Time:')
        self.time_slider = widgets.SelectionRangeSlider(options=time_options, index=(0, len(time_options) - 1), description='Time Range:', layout={'width': '100%'})

        # Weekdays/months
        weekday_options = ['All','Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
        self.weekday_options = weekday_options
        self.weekday_multi = widgets.SelectMultiple(options=weekday_options, value=tuple(weekday_options[1:]), description='Weekdays:', layout={'height': '80px'})

        month_options = ['All','January','February','March','April','May','June','July','August','September','October','November','December']
        self.month_options = month_options
        self.month_multi = widgets.SelectMultiple(options=month_options, value=tuple(month_options[1:]), description='Months:', layout={'height': '100px'})

        # Criteria & C Target (searchable + invert)
        unique_criteria = sorted(df['Criteria'].dropna().astype(str).unique().tolist())
        self.unique_criteria = unique_criteria
        self.criteria_search = widgets.Text(placeholder='Search Criteria')
        self.criteria_multi = widgets.SelectMultiple(options=['All'] + unique_criteria, value=tuple(unique_criteria), description='Criteria:', layout={'height': '120px'})
        self.criteria_invert_button = widgets.Button(description='Invert Selection', layout={'width': '120px'})

        unique_cTarget = sorted(df['C Target'].fillna("NA").astype(str).dropna().unique().tolist())
        self.unique_cTarget = unique_cTarget
        self.cTarget_search = widgets.Text(placeholder='Search C Target')
        self.cTarget_multi = widgets.SelectMultiple(options=['All'] + unique_cTarget, value=tuple(unique_cTarget), description='C Target:', layout={'height': '120px'})
        self.cTarget_invert_button = widgets.Button(description='Invert Selection', layout={'width': '120px'})

        # Post-aggregation sliders (bounds set after first summary)
        self.trade_count_slider = widgets.IntRangeSlider(value=[0, 0], min=0, max=0, description='Trades Range:')
        self.win_rate_slider    = widgets.FloatRangeSlider(value=[0.0, 1.0], min=0.0, max=1.0, step=0.01, description='Win Rate Range:')
        self.win_days_slider    = widgets.IntRangeSlider(value=[0, 0], min=0, max=0, description='Win Days Range:')
        self.pl_slider          = widgets.FloatRangeSlider(value=[0.0, 0.0], min=0.0, max=0.0, step=1.0, description='Realized P/L Range:', layout={'width': '98%'})

        # Export buttons
        self.export_html_btn = widgets.Button(description='💾 Save Chart (HTML)', layout={'width': '180px'})
        self.export_csv_btn  = widgets.Button(description='💾 Save Summary (CSV)', layout={'width': '200px'})
        self.msg_area = widgets.HTML(value='')

        # Output area
        self.output = widgets.Output()

        # Wire events
        self._wire_events()

    # ---------- Event helpers ----------
    def _wire_events(self):
        # Visibility
        self.chart_dropdown.observe(self._update_group_visibility, names='value')

        # Triggers for full recompute
        for w in [self.chart_dropdown, self.group1_dropdown, self.group2_dropdown,
                  self.weekday_multi, self.month_multi, self.criteria_multi, self.cTarget_multi]:
            w.observe(self._on_select, names='value')

        # Date/time sync between picker and slider
        self.date_slider.observe(self._on_date_slider_change, names='value')
        self.start_date_picker.observe(self._on_date_picker_change, names='value')
        self.end_date_picker.observe(self._on_date_picker_change, names='value')

        self.time_slider.observe(self._on_time_slider_change, names='value')
        self.start_time_picker.observe(self._on_time_picker_change, names='value')
        self.end_time_picker.observe(self._on_time_picker_change, names='value')

        # Search + invert
        self.criteria_search.observe(partial(self._update_searchable_multiselect, multiselect_widget=self.criteria_multi, full_list=self.unique_criteria), names='value')
        self.cTarget_search.observe(partial(self._update_searchable_multiselect, multiselect_widget=self.cTarget_multi, full_list=self.unique_cTarget), names='value')
        self.criteria_invert_button.on_click(partial(self._invert_multiselect_selection, multiselect_widget=self.criteria_multi, full_list=self.unique_criteria))
        self.cTarget_invert_button.on_click(partial(self._invert_multiselect_selection, multiselect_widget=self.cTarget_multi, full_list=self.unique_cTarget))

        # Post-filter sliders
        for w in [self.trade_count_slider, self.win_rate_slider, self.pl_slider, self.win_days_slider]:
            w.observe(self._on_select_post_filter, names='value')

        # Exports
        self.export_html_btn.on_click(self._save_chart_html)
        self.export_csv_btn.on_click(self._save_summary_csv)

    # ---------- UI logic ----------
    def _update_group_visibility(self, change=None):
        self.group2_dropdown.layout.display = 'block' if self.chart_dropdown.value == 'heatmap' else 'none'

    def _update_searchable_multiselect(self, change, multiselect_widget, full_list):
        q = (change['new'] or '').lower()
        if not q:
            options = ['All'] + full_list
            selections = tuple(full_list)
        else:
            filtered = [opt for opt in full_list if q in opt.lower()]
            options = ['All'] + filtered
            selections = tuple(filtered)
        multiselect_widget.options = options
        multiselect_widget.value = selections

    def _invert_multiselect_selection(self, button, multiselect_widget, full_list):
        current = set(multiselect_widget.value)
        inverse = [opt for opt in full_list if opt not in current]
        multiselect_widget.options = ['All'] + full_list
        multiselect_widget.value = tuple(inverse)

    # ---------- Filters ----------
    def _resolve_selection(self, widget, all_values):
        selected = list(widget.value)
        if (not selected) or ('All' in selected):
            return all_values
        return selected

    def _filter_dataframe(self) -> pd.DataFrame:
        df = self.df
        # Date filter
        start_date = pd.to_datetime(self.start_date_picker.value)
        end_date   = pd.to_datetime(self.end_date_picker.value)
        df_f = df[(pd.to_datetime(df['Start Date']) >= start_date) & (pd.to_datetime(df['Start Date']) <= end_date)]

        # Time filter (with overnight support)
        st = self.start_time_picker.value
        et = self.end_time_picker.value
        if st <= et:
            mask_time = (df_f['Start Time'] >= st) & (df_f['Start Time'] <= et)
        else:
            # Overnight window (e.g., 22:00 → 03:00): allow times >= st OR <= et
            mask_time = (df_f['Start Time'] >= st) | (df_f['Start Time'] <= et)
        df_f = df_f[mask_time]

        # Weekday
        weekdays = self._resolve_selection(self.weekday_multi, self.weekday_options[1:])
        df_f = df_f[df_f['Start_Weekday'].isin(weekdays)]

        # Month
        months = self._resolve_selection(self.month_multi, self.month_options[1:])
        df_f = df_f[df_f['Month'].isin(months)]

        # Criteria
        crits = self._resolve_selection(self.criteria_multi, self.unique_criteria)
        df_f = df_f[df_f['Criteria'].astype(str).isin(crits)]

        # C Target
        cts = self._resolve_selection(self.cTarget_multi, self.unique_cTarget)
        df_f = df_f[df_f['C Target'].fillna("NA").astype(str).isin(cts)]
        return df_f

    # # ---------- Slider safety ----------
    # #@staticmethod
    # def _update_slider_range( slider, new_min, new_max):
    #     # Handle empty or NaN
    #     if pd.isna(new_min) or pd.isna(new_max):
    #         new_min, new_max = 0, 1  # default safe range

    #     if isinstance(slider, widgets.IntRangeSlider):
    #         if new_min > slider.max:
    #             slider.max = int(new_min)
    #             slider.min = int(new_max)
    #         else:
    #           slider.min = int(new_min)
    #           slider.max = int(new_max)
    #         slider.value = (int(new_min), int(new_max))
    #     else:
    #         if new_min > slider.max:
    #           slider.max = float(new_min)
    #           slider.min = float(new_max)
    #         else:
    #           slider.min = float(new_min)
    #           slider.max = float(new_max)
    #         slider.value = (float(new_min), float(new_max))

    @staticmethod
    def _update_slider_range( slider, new_min, new_max):
      if new_min > slider.max:
        slider.max = new_max
        slider.min = new_min
      else:
        slider.min = new_min
        slider.max = new_max
      slider.value = (new_min, new_max)

    # ---------- Recompute ----------
    def _update_summary_and_sliders(self, df_filtered: pd.DataFrame) -> pd.DataFrame:
        # Detach to avoid recursive triggers
        for w in [self.trade_count_slider, self.win_rate_slider, self.pl_slider, self.win_days_slider]:
            w.unobserve(self._on_select_post_filter, names='value')

        keys = [self.group1_dropdown.value]
        if self.chart_dropdown.value == 'heatmap':
            keys.append(self.group2_dropdown.value)

        self.last_filtered_df = df_filtered
        summary_df = calculate_trade_metrics_updated(df_filtered, keys, self.chart_dropdown.value)
        self.summary_df = summary_df
        self.last_summary = summary_df.copy()

        display(self.summary_df.head(12))

        # Update slider bounds safely
        TradingDashboard._update_slider_range(self.trade_count_slider, summary_df['Total_Trades'].min(), summary_df['Total_Trades'].max())
        wr_min, wr_max = summary_df['Win_Rate'].min(), summary_df['Win_Rate'].max()
        # keep 4 dp in domain, but display 2 dp in hover
        self.win_rate_slider.min = float(np.floor(wr_min*10000)/10000) if pd.notna(wr_min) else 0.0
        self.win_rate_slider.max = float(np.ceil(wr_max*10000)/10000) if pd.notna(wr_max) else 1.0
        self.win_rate_slider.value = (self.win_rate_slider.min, self.win_rate_slider.max)

        pl_min, pl_max = summary_df['Realized_Profit_Loss'].min(), summary_df['Realized_Profit_Loss'].max()
        self.pl_slider.min = float(np.floor(pl_min)) if pd.notna(pl_min) else 0.0
        self.pl_slider.max = float(np.ceil(pl_max))  if pd.notna(pl_max) else 0.0
        self.pl_slider.value = (self.pl_slider.min, self.pl_slider.max)

        # Win days slider optional (kept for parity; bound by observed values)
        TradingDashboard._update_slider_range(self.win_days_slider, summary_df['Total_Win_Days'].min(), summary_df['Total_Win_Days'].max())

        # Reattach
        for w in [self.trade_count_slider, self.win_rate_slider, self.pl_slider, self.win_days_slider]:
            w.observe(self._on_select_post_filter, names='value')

        return summary_df

    def _post_filter_summary_df(self) -> pd.DataFrame:
        if self.summary_df.empty:
            return self.summary_df
        tmin, tmax = self.trade_count_slider.value
        wrmin, wrmax = self.win_rate_slider.value
        plmin, plmax = self.pl_slider.value
        # Optional win day bounds
        wdmin, wdmax = self.win_days_slider.value

        sdf = self.summary_df[
            (self.summary_df['Total_Trades'] >= tmin) & (self.summary_df['Total_Trades'] <= tmax) &
            (self.summary_df['Win_Rate'] >= float(np.floor(wrmin*10000)/10000)) & (self.summary_df['Win_Rate'] <= float(np.ceil(wrmax*10000)/10000)) &
            (self.summary_df['Realized_Profit_Loss'] >= float(np.floor(plmin))) & (self.summary_df['Realized_Profit_Loss'] <= float(np.ceil(plmax))) &
            (self.summary_df['Total_Win_Days'] >= wdmin) & (self.summary_df['Total_Win_Days'] <= wdmax)
        ]
        return sdf

    # ---------- Update handlers ----------
    def _on_time_slider_change(self, change):
        self.start_time_picker.value, self.end_time_picker.value = change['new']
        self._on_select()

    def _on_time_picker_change(self, change):
        if self.start_time_picker.value and self.end_time_picker.value:
            self.time_slider.value = (self.start_time_picker.value, self.end_time_picker.value)
            self._on_select()

    def _on_date_slider_change(self, change):
        self.start_date_picker.value, self.end_date_picker.value = change['new']
        self._on_select()

    def _on_date_picker_change(self, change):
        if self.start_date_picker.value and self.end_date_picker.value:
            self.date_slider.value = (self.start_date_picker.value, self.end_date_picker.value)
            self._on_select()

    def _on_select(self, change=None):
        self.output.clear_output(wait=True)
        with self.output:
            try:
                df_filtered = self._filter_dataframe()
                if df_filtered.empty:
                    print("⚠️ No data after filtering. Try different filters.")
                    return
                summary = self._update_summary_and_sliders(df_filtered)
                sdf = self._post_filter_summary_df()
                keys = [self.group1_dropdown.value] if self.chart_dropdown.value != 'heatmap' else [self.group1_dropdown.value, self.group2_dropdown.value]
                fig = generate_visualizations_updated(sdf, keys, self.chart_dropdown.value)
                self.last_fig = fig
                self.last_summary = summary
                # fig.show()
            except Exception:
                traceback.print_exc()

    def _on_select_post_filter(self, change=None):
        self.output.clear_output(wait=True)
        with self.output:
            try:
                if self.summary_df.empty:
                    print("⚠️ No summary data to filter.")
                    return
                sdf = self._post_filter_summary_df()
                keys = [self.group1_dropdown.value] if self.chart_dropdown.value != 'heatmap' else [self.group1_dropdown.value, self.group2_dropdown.value]
                fig = generate_visualizations_updated(sdf, keys, self.chart_dropdown.value)
                self.last_fig = fig
                print("✅ Chart updated. Call dashboard.last_fig.show() to display it again.")
                # fig.show()
            except Exception:
                traceback.print_exc()

    # ---------- Exports ----------
    def _save_chart_html(self, _):
        if self.last_fig is None:
            self.msg_area.value = '<span style="color:red">No chart to save.</span>'
            return
        fname = 'trading_chart.html'
        self.last_fig.write_html(fname, include_plotlyjs='cdn')
        self.msg_area.value = f'<span>Saved: <code>{fname}</code></span>'

    def _save_summary_csv(self, _):
        if self.last_summary is None or isinstance(self.last_summary, pd.DataFrame) and self.last_summary.empty:
            self.msg_area.value = '<span style="color:red">No summary to save.</span>'
            return
        fname = 'trading_summary.csv'
        self.last_summary.to_csv(fname, index=False)
        self.msg_area.value = f'<span>Saved: <code>{fname}</code></span>'

    # ---------- Layout ----------
    def display(self):
        self._update_group_visibility()
        ui = widgets.VBox([
            self.chart_dropdown,
            widgets.HBox([self.group1_dropdown, self.group2_dropdown]),
            widgets.VBox([self.date_slider, widgets.HBox([self.start_date_picker, self.end_date_picker])]),
            widgets.VBox([self.time_slider, widgets.HBox([self.start_time_picker, self.end_time_picker])]),
            widgets.HBox([
                self.weekday_multi,
                self.month_multi,
                widgets.VBox([self.criteria_search, self.criteria_multi, self.criteria_invert_button]),
                widgets.VBox([self.cTarget_search, self.cTarget_multi, self.cTarget_invert_button]),
            ]),
            widgets.Label('Post-Aggregation Filters:'),
            self.trade_count_slider,
            self.win_rate_slider,
            self.pl_slider,
            self.win_days_slider,
            widgets.HBox([self.export_html_btn, self.export_csv_btn, self.msg_area]),
            self.output
        ])
        display(ui)
        self._on_select()

In [ ]:
def launch_chart_selector_updated(df: pd.DataFrame):
    """Drop-in replacement for your existing entry-point (no API change)."""
    dashboard = TradingDashboard(df)
    dashboard.display()
    return dashboard  # so you can access .last_fig / .last_summary later

In [ ]:
main_df = pd.read_excel('SSL_T.xlsx')
df = prepare_dataframe(main_df)
dashboard = launch_chart_selector_updated(df)

/tmp/ipython-input-3751155631.py:33: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



In [ ]:
dashboard.last_fig.show()

In [ ]:
dashboard.last_filtered_df.head()

,S/l,Start Date,Start Time,W-D-4H-1H-15M-5M-1M,Criteria,Confluence,Candle/Days,Signal,Entry,Stop Loss,...,Month,Week,Year,Month_Year,Start_Weekday,Date,Day_of_Month,Week_of_Month,Week_of_Month_Name,Win R
0,1,2024-10-23,09:15:00,NaN,LG PDL,NaN,1.0,GDoji LD,2.360,2.352,...,October,43,2024,October-2024-10,Wednesday,2024-10-23,23,4,4-October,5.00
1,2,2024-10-23,13:00:00,NaN,LG RGC,NaN,2.0,BBR LD,2.418,2.411,...,October,43,2024,October-2024-10,Wednesday,2024-10-23,23,4,4-October,7.00
2,3,2024-10-23,15:45:00,NaN,LG PDH,NaN,13.0,GDoji LD,2.394,2.376,...,October,43,2024,October-2024-10,Wednesday,2024-10-23,23,4,4-October,6.56
3,4,2024-10-23,21:15:00,NaN,LG PDL,NaN,5.0,BBG LW,2.534,2.614,...,October,43,2024,October-2024-10,Wednesday,2024-10-23,23,4,4-October,NaN
4,5,2024-10-23,22:00:00,NaN,LG RGC,NaN,2.0,RHammer,2.495,2.486,...,October,43,2024,October-2024-10,Wednesday,2024-10-23,23,4,4-October,NaN


In [ ]:
dashboard.last_summary.head()

,15Min,Total_Trades,Total_Win,Total_Loss,Total_Profit,Total_Loss_Amount,Total_Fee,Total_Time,Average_Time,Realized_Profit_Loss,...,Max_Profit,Max_Loss,R,Average_Win_R,Total_Win_Days,Total_Loss_Days,Average_Win_Days_Profit,Average_Daily_Profit,Max_Daily_Profit,Win_Rate
0,00:00,27,12,15,2531.00,655.0,356.585901,26.000000,0.962963,1519.414099,...,350.225875,-63.963250,5.202593,5.970000,12,15,197.364184,56.274596,350.225875,0.4444
1,00:15,51,25,26,4342.13,1177.0,622.313371,51.000000,1.000000,2542.816629,...,425.669250,-64.113443,3.844706,3.736800,25,26,160.589829,49.859150,425.669250,0.4902
2,00:30,29,13,16,2388.64,708.0,336.821968,27.833333,0.959770,1343.818032,...,347.805048,-64.385577,4.377931,3.876154,13,15,172.305837,47.993501,347.805048,0.4483
3,00:45,32,9,23,1739.78,961.0,410.510869,25.333333,0.791667,368.269131,...,298.834448,-64.668382,4.854687,5.581111,9,23,179.428171,11.508410,298.834448,0.2812
4,01:00,28,15,13,2450.68,536.0,343.474756,29.333333,1.047619,1571.205244,...,248.802000,-62.462500,4.544643,4.316000,15,13,151.112650,56.114473,248.802000,0.5357


In [ ]:
import pandas as pd
import numpy as np
import calendar
import plotly.graph_objects as go

def generate_trade_calendar(df, year, month):
    # Ensure Date is datetime
    df['Date'] = pd.to_datetime(df['Date'])

    # Filter for selected month
    month_df = df[(df['Date'].dt.year == year) & (df['Date'].dt.month == month)]

    # Aggregate daily stats
    daily_stats = month_df.groupby('Date').agg(
        total_trades=('Profit', 'count'),  # change column name as needed
        wins=('Profit', lambda x: (x > 0).sum()),
        losses=('Profit', lambda x: (x <= 0).sum()),
        total_profit=('Profit', 'sum'),
        realized_profit=('Profit/Loss', 'sum'),  # change as needed
        max_profit=('Profit', 'max')
    ).reset_index()

    daily_stats['win_rate'] = (daily_stats['wins'] / daily_stats['total_trades'] * 100).round(1)
    # Create month layout
    cal = calendar.Calendar(firstweekday=6)  # Sunday start
    month_days = list(cal.itermonthdates(year, month))

    # Create weekly totals
    weekly_totals = []
    for week in calendar.Calendar(firstweekday=6).monthdatescalendar(year, month):
        week_df = daily_stats[daily_stats['Date'].isin(week)]
        weekly_totals.append({
            'profit': week_df['total_profit'].sum(),
            'wins': week_df['wins'].sum(),
            'losses': week_df['losses'].sum(),
            'win_days': (week_df['realized_profit'] > 0).sum(),   # NEW
            'loss_days': (week_df['realized_profit'] < 0).sum(), # NEW
            'realized_profit': week_df['realized_profit'].sum(),
            'days': week_df.shape[0]
        })

    # --- In monthly_stats ---
    monthly_stats = pd.DataFrame({
        'Total Trades': [len(month_df)],
        'Wins': [len(month_df[month_df['Profit'] > 0])],
        'Losses': [len(month_df[month_df['Profit'] <= 0])],
        'Win Days': [(daily_stats['realized_profit'] > 0).sum()],      # NEW
        'Loss Days': [(daily_stats['realized_profit'] < 0).sum()],   # NEW
        'Total Profit': [month_df['Profit'].sum()],
        'Realized Profit/Loss': [month_df['Profit/Loss'].sum()],
        'Win Rate': [round((len(month_df[month_df['Profit/Loss'] > 0]) / len(month_df) * 100) if len(month_df) > 0 else 0, 1)],
        'Max Profit': [df['Profit/Loss'].max()]
    }).fillna(0)
    # print(monthly_stats)
    # Prepare Plotly figure
    fig = go.Figure()

    cell_w, cell_h = 1, 1
    week_num = 0

    for row_idx, week in enumerate(calendar.Calendar(firstweekday=6).monthdatescalendar(year, month)):
        for col_idx, day in enumerate(week):
            x0 = col_idx * cell_w
            y0 = -row_idx * cell_h
            x1 = x0 + cell_w
            y1 = y0 - cell_h

            # Empty days
            if day.month != month:
                color = "#222"
                fig.add_shape(type="rect", x0=x0, y0=y0, x1=x1, y1=y1,
                              line=dict(color="#444"), fillcolor=color)
                continue

            day_data = daily_stats[daily_stats['Date'].astype(str)== str(day)]

            if not day_data.empty:
                profit = day_data['realized_profit'].values[0]
                trades = int(day_data['total_trades'].values[0])
                win_rate = day_data['win_rate'].values[0]

                color = "rgba(0,100,0,0.7)" if profit > 0 else "rgba(139,0,0,0.7)"
                text_color = "white"
                hovertext = (
                    f"<b>{day.strftime('%Y-%m-%d')}</b><br>"
                    f"Profit: {profit:,.2f}<br>"
                    f"Trades: {trades}<br>"
                    f"Win Rate: {win_rate}%<br>"
                    f"Wins: {day_data['wins'].values[0]}, Losses: {day_data['losses'].values[0]}<br>"
                    f"Realized: {day_data['realized_profit'].values[0]:,.2f}<br>"
                    f"Max Profit: {day_data['max_profit'].values[0]:,.2f}"
                )
                d_p_label = int(profit) if profit <1000 else f"{profit/1000:.2f}K"
                d_label = f"<b>{day.day}</b>"
                label = f"<br>${d_p_label}<br>{trades} trades<br>W:{day_data['wins'].values[0]} | L:{day_data['losses'].values[0]}<br>{win_rate:.0f}%"
            else:
                color = "#333"
                text_color = "#888"
                label = f"<b>{day.day}</b>"
                hovertext = f"{day.strftime('%Y-%m-%d')}<br>No trades"

            # Draw cell
            fig.add_shape(type="rect", x0=x0, y0=y0, x1=x1, y1=y1,
                          line=dict(color="#555"), fillcolor=color)
            fig.add_trace(go.Scatter(
                x=[(x0+x1)/2], y=[(y0+y1)/2],
                text=[label],
                mode="text",
                textfont=dict(color=text_color, size=12),
                hovertext=hovertext,
                hoverinfo="text"
            ))
            fig.add_trace(go.Scatter(
                x=[x1-.1], y=[y1+.8],
                text=[d_label],
                mode="text",
                textfont=dict(color=text_color, size=12),
                hovertext=hovertext,
                hoverinfo="text"
            ))

        # Weekly summary column
        week_profit = weekly_totals[row_idx]['realized_profit']
        week_days = weekly_totals[row_idx]['days']
        w_p_label = int(week_profit) if week_profit <1000 else f"{week_profit/1000:.2f}K"
        week_label = (
            f"<b>Week {row_idx+1}</b><br>${w_p_label}<br>{week_days} days<br>"
            f"Wins: {weekly_totals[row_idx]['wins']} | Losses: {weekly_totals[row_idx]['losses']}<br>"
            f"W Days: {weekly_totals[row_idx]['win_days']} | LDays: {weekly_totals[row_idx]['loss_days']}"
        )
        week_color = "rgba(0,100,0,0.8)" if week_profit > 0 else "rgba(139,0,0,0.8)"
        x0 = 7 * cell_w
        x1 = x0 + cell_w
        y0 = -row_idx * cell_h
        y1 = y0 - cell_h

        fig.add_shape(type="rect", x0=x0, y0=y0, x1=x1, y1=y1,
                      line=dict(color="#555"), fillcolor=week_color)
        fig.add_trace(go.Scatter(
            x=[(x0+x1)/2], y=[(y0+y1)/2],
            text=[week_label],
            mode="text",
            textfont=dict(color="white", size=12),
            hoverinfo="skip"
        ))

    # Weekday headers
    for i, wd in enumerate(["Sun","Mon","Tue","Wed","Thu","Fri","Sat","Week Total"]):
        fig.add_trace(go.Scatter(
            x=[i*cell_w + cell_w/2], y=[cell_h/3],
            text=[f"<b>{wd}</b>"], mode="text",
            textfont=dict(size=14, color="white"),
            hoverinfo="skip"
        ))

    m_p_label = int(monthly_stats['Total Profit'][0]) if monthly_stats['Total Profit'][0] <1000 else f"{monthly_stats['Total Profit'][0]/1000:.2f}K"
    m_rp_label = int(monthly_stats['Realized Profit/Loss'][0]) if monthly_stats['Realized Profit/Loss'][0] <1000 else f"{monthly_stats['Realized Profit/Loss'][0]/1000:.2f}K"
    summary_text = (
        f"<b>{calendar.month_name[month]} {year} Summary</b><br>"
        f"Trades: {monthly_stats['Total Trades'][0]} | Wins: {monthly_stats['Wins'][0]} | "
        f"Losses: {monthly_stats['Losses'][0]}<br>"
        f"Win Days: {monthly_stats['Win Days'][0]} | Loss Days: {monthly_stats['Loss Days'][0]}<br>"
        f"Total Profit: {m_p_label} | "
        f"Realized: ${m_rp_label} | "
        f"Win Rate: {monthly_stats['Win Rate'][0]}%"
    )
    print(summary_text)

    fig.add_trace(go.Scatter(
            x=[3.5],  # Center above calendar (7 columns / 2)
            y=[cell_h * 1],  # slightly above first row
            text=[summary_text],
            mode="text",
            textfont=dict(color="skyblue", size=16),
            hoverinfo="skip"
        ))
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    fig.update_layout(
        plot_bgcolor="#000",
        paper_bgcolor="#000",
        height=800,
        margin=dict(l=20, r=20, t=20, b=20)
    )

    return fig


In [ ]:

df = prepare_dataframe(main_df)
fig = generate_trade_calendar(df, 2025, 8)
fig.show()

/tmp/ipython-input-3751155631.py:33: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

/tmp/ipython-input-390514240.py:31: FutureWarning:

The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.



<b>August 2025 Summary</b><br>Trades: 79 | Wins: 41 | Losses: 38<br>Win Days: 10 | Loss Days: 1<br>Total Profit: 5.80K | Realized: $3.04K | Win Rate: 51.9%
